import 与数据导入部分

In [1]:
import numpy as np
import pandas as pd
import re
import warnings
#from scipy.stats import linregress
#from joblib import Parallel, delayed
import warnings

train=pd.read_csv('train.csv')

TARGET_COL = 'market_forward_excess_returns'
EXCLUDE_COLS = ['date_id', 'forward_returns', 'risk_free_rate', TARGET_COL]

数据预处理

In [2]:
def get_feature_group(prefix):
        return [c for c in train.columns if re.match(f'^{prefix}[0-9]+$', c)] ##正则筛选特定前缀的列名，“[0-9]+”匹配一或多个数字
groups = {
    'D': get_feature_group('D'),
    'E': get_feature_group('E'),
    'I': get_feature_group('I'),
    'M': get_feature_group('M'),
    'P': get_feature_group('P'),
    'S': get_feature_group('S'),
    'V': get_feature_group('V'),
    }

print(groups)

{'D': ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9'], 'E': ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9'], 'I': ['I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9'], 'M': ['M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'], 'P': ['P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9'], 'S': ['S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9'], 'V': ['V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9']}


In [ ]:




def _calculate_single_column_stress(
    vol_series: pd.Series,
    window: int
) -> pd.Series:
    """计算单列市场压力指数"""
    if vol_series.empty or vol_series.isna().all():
        return pd.Series(0.5, index=vol_series.index)
    
    # 1. 波动率历史分位数（核心）
    rolling_quantile = (
        vol_series
        .expanding(min_periods=window)
        .apply(lambda x: (x[-1] > x[:-1]).mean() if len(x) > 1 else 0.5, raw=False)
        .fillna(0.5)  # 初始值
    )
    
    # 2. 波动率斜率（加速上升）
    vol_slope = (
        vol_series
        .rolling(window=5, min_periods=3)
        .apply(lambda x: linregress(range(len(x)), x)[0] if len(x) > 3 else 0, raw=False)
    )
    vol_std = vol_series.rolling(window).std().fillna(1)
    vol_slope_norm = vol_slope / vol_std
    
    # 3. 融合压力指数
    stress_index = (
        rolling_quantile * 0.7 + 
        ((vol_slope_norm > 0) & (vol_slope_norm > vol_slope_norm.rolling(10).quantile(0.7))).astype(float) * 0.3
    )
    
    return stress_index.clip(0, 1).fillna(0.5)

def vol_midium_missing(
    vol_df: pd.DataFrame,
    pressure_window: int = 63,
    lookback_window: int = 21,
    parallel: bool = False,  # 默认关闭并行
    n_jobs: int = -1
) -> pd.DataFrame:
    """
    处理波动率特征的中等缺失值
    严格保持原始索引和列名
    """
    # 验证输入
    if not isinstance(vol_df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame")
    if vol_df.empty:
        return vol_df.copy()
    
    original_index = vol_df.index.copy()
    original_columns = vol_df.columns.tolist()
    
    # 安全处理单列情况
    if len(original_columns) == 1:
        parallel = False
    
    def process_single_column(col_name: str) -> pd.Series:
        """处理单个波动率列，返回带原始索引的Series"""
        if col_name not in vol_df.columns:
            raise KeyError(f"Column '{col_name}' not found in input DataFrame")
        
        col_series = vol_df[col_name].copy()
        original_name = col_series.name
        
        # 阶段1：时序局部填充
        ewm_filled = col_series.fillna(
            col_series.ewm(span=lookback_window, min_periods=1).median()
        )
        
        # 阶段2：计算市场压力
        market_stress = _calculate_single_column_stress(col_series, pressure_window)
        
        # 阶段3：全局统计填充（按压力分组）
        if ewm_filled.isna().any():
            stress_threshold = market_stress.quantile(0.7)
            high_stress_mask = market_stress > stress_threshold
            
            # 高压力区域：用90分位数
            high_stress_vol = col_series[high_stress_mask & ~col_series.isna()]
            high_fill_val = (
                high_stress_vol.quantile(0.9) if len(high_stress_vol) > 5 
                else col_series.quantile(0.75)
            )
            
            # 低压力区域：用50分位数
            low_stress_vol = col_series[~high_stress_mask & ~col_series.isna()]
            low_fill_val = (
                low_stress_vol.quantile(0.5) if len(low_stress_vol) > 5 
                else col_series.quantile(0.5)
            )
            
            # 应用填充
            ewm_filled.loc[high_stress_mask & ewm_filled.isna()] = high_fill_val
            ewm_filled.loc[~high_stress_mask & ewm_filled.isna()] = low_fill_val
        
        # 最终填充
        final_filled = ewm_filled.ffill().bfill()
        
        return pd.Series(
            data=final_filled.values,
            index=original_index,
            name=original_name,
            dtype=final_filled.dtype
        )
    
    # 处理所有列
    if parallel and len(original_columns) > 1:
        # 并行处理
        try:
            results = Parallel(n_jobs=n_jobs)(
                delayed(process_single_column)(col) for col in original_columns
            )
            # 重建DataFrame
            filled_data = {col: res for col, res in zip(original_columns, results)}
            result = pd.DataFrame(filled_data, index=original_index)
        except Exception as e:
            print(f"ParallelGroup processing failed: {str(e)}. Falling back to sequential.")
            parallel = False
    
    if not parallel or 'result' not in locals():
        # 顺序处理（更安全）
        filled_data = {}
        for col in original_columns:
            try:
                filled_data[col] = process_single_column(col)
            except Exception as e:
                print(f"Error processing column '{col}': {str(e)}")
                # 回退方案
                col_series = vol_df[col].copy()
                filled_data[col] = col_series.ffill().bfill()
        
        # 重建DataFrame
        result = pd.DataFrame(filled_data, index=original_index)
    
   
    result = result[original_columns]  # 保持原始列顺序
    result.index = original_index      # 确保索引正确
    
    return result

def handle_missing(train):
    """分层次与类别处理缺失值，全部的特征包括[D1,D2,D3,...D9   二进制特征
                  E1,E2,E3...E20  宏观经济特征
                  I1,I2,I3...I9    利率特征，滞后
                  M1,M2,M3...M18  市场技术特征，滞后
                  P1,P2,P3...P13  价格，估值特征
                  S1,S2,S3...S12   情感特征
                  V1,V2,V3...V13   波动率，滞后 
                  MOM,     动量特征 实际没有自己创建         ]
        高缺失率（>50%):直接删除，中等（5-50）使用时间序列特性填充，低（<5%)前向或者迭代填充
    """
    
    missing_ratio = train.isna().mean()
    too_missing = missing_ratio[missing_ratio > 0.5].index.tolist()
    train.drop(columns=too_missing, inplace=True, errors='ignore')

    low_missing_cols = set(missing_ratio[(missing_ratio > 0) & (missing_ratio < 0.05)].index.tolist())
    medium_missing_cols = set(missing_ratio[(missing_ratio >= 0.05) & (missing_ratio <= 0.5)].index.tolist())

    
    # 对低缺失率以及中缺失率数据按组进行缺失处理
    for gname, cols in groups.items():
        valid_cols1=set(cols) & low_missing_cols
        valid_cols2=set(cols) & medium_missing_cols

        for col in valid_cols1:
            
            if gname == 'D':
                train[col] = train[col].fillna(0)
            elif gname in ['E', 'I','P','V']:
                train[col] = train[col].ffill().fillna(train[col].median())
            elif gname == 'V':
                
                train[col] = train[col].fillna(train[col].rolling(window=21,min_periods=1).median()).ffill()
            elif gname == 'M':
                train[col] = train[col].ffill().fillna(train[col].mean())
            elif gname == 'S':
                col_series = train[col].copy()
                col_series = col_series.fillna(col_series.rolling(5, min_periods=1).mean())
                col_series = col_series.ffill(limit=5)
                col_series = col_series.ffill().fillna(col_series.mean())
                train[col] = col_series
        for col in valid_cols2:
            
            if gname == 'D':
                train[col] = train[col].fillna(0)
            elif gname in ['E', 'I','P']:
                train[col] = train[col].ffill().bfill()

            elif gname == 'V':
                col_df = train[[col]].ffill()
                filled_col = vol_midium_missing(
                    col_df, 
                    pressure_window=63,
                    lookback_window=21,
                    parallel=False

                )
                train[col] = filled_col[col]
                # train[cols] = vol_midium_missing(train[cols].ffill())
            elif gname == 'M':
                train[col] = train[col].ffill().fillna(train[col].mean())
            elif gname == 'S':
                col_series = train[col].copy()
                col_series = col_series.fillna(col_series.rolling(5, min_periods=1).mean())
                col_series = col_series.ffill(limit=5)
                col_series = col_series.ffill().fillna(col_series.mean())
                train[col] = col_series
             
    return train




def handle_outliers(df, clip=1):
    """处理异常值"""
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns 
                    if c not in EXCLUDE_COLS]
    for col in numeric_cols:
        low = df[col].quantile(clip/100)
        high = df[col].quantile(1-clip/100)
        df[col] = df[col].clip(low, high)
    return df

print("Preprocessing training data...")
# train = train.iloc[1005:].reset_index(drop=True)

train = handle_missing(train)
train = handle_outliers(train)

print(f"Train shape: {train.shape}")

Preprocessing training data...
Error processing column 'V5': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V13': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V11': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V2': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V3': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V4': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V12': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V8': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V1': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V7': 'ExponentialMovingWindow' object has no attribute 'median'
Error processing column 'V6': 'ExponentialMovingWindow' object has no attrib

In [4]:
print(f"nonsum:{train.notnull().sum()}")

nonsum:date_id                          9021
D1                               9021
D2                               9021
D3                               9021
D4                               9021
                                 ... 
V7                               9021
V8                               9021
forward_returns                  9021
risk_free_rate                   9021
market_forward_excess_returns    9021
Length: 90, dtype: int64


特征工程部分

In [ ]:
def create_finance_features(df):
    """创建特征，其中根据EDA的数据集，特征与目标斯皮尔曼相关系数以及共线性分析
    全部的特征包括[D1,D2,D3,...D9   二进制特征
                  E1,E2,E3...E20  宏观经济特征
                  I1,I2,I3...I9    利率特征
                  M1,M2,M3...M18  市场动态技术特征，滞后
                  P1,P2,P3...P13  价格，估值特征
                  S1,S2,S3...S12   情感特写,短期滞后，123加权
                  V1,V2,V3...V13   波动率，滞后
                  创建mom动量特征，mom5，mom21           ]
    其中，前十和最后十名的相关特征分别为[M1,V13,V10,S5,V7,E19,V9,S12,D2,D1],[E12,S8,P8,M12,I2,S3,E11,P5,S2,M4]
    共线性最高的十位特征为[I5,I9,M14,E2,P10,E3,I7,P11,P8,I8]
    同时我们通过检验证明了该时间序列基本满足平稳性(stationary)
    """
    
    #关于滞后特征
    '''
    # 动量特征
    df['mom_5d'] = df[TARGET_COL].rolling(window=5, min_periods=1).sum()
    df['mom_21d'] = df[TARGET_COL].rolling(window=21, min_periods=1).sum()
    
    for gname , cols in groups.items():
        if gname =='S':
            for col in cols:
                shifted_col = df[col].shift(1)
            
            # 计算加权滞后特征（只使用历史数据）
                df[f"lag_{col}"] = (
                    0.5 * shifted_col +  
                    0.3 * shifted_col.rolling(window=3, min_periods=1).mean() +  
                    0.2 * shifted_col.rolling(window=5, min_periods=1).mean()    
                )
                
                del shifted_col

            
        elif gname == 'M':

        elif gname == 'V':
        '''
        #这里被提示了性能问题，内存碎片化严重，因为循环中多次插入新列df[f"lag_{col}]=...，因此采取优化
        #方案如下
      # === 1. 定义所有需要管理的特征列名 ===
    # 情绪特征 (lag_)
    lag_features = [
        f"lag_{col}" 
        for col in groups.get('S', []) 
        if col in df.columns
    ]
    
    # 动量特征 (固定名称)
    df['hist_returns'] = df['forward_returns'].shift(1)
    mom_features = ['mom_21d', 'mom_42d','mom_63d','mom_126']
    
    # 合并所有需要管理的特征
    all_managed_features = lag_features + mom_features
    
    # === 2. 删除所有旧特征（幂等性核心） ===
    existing_features = [f for f in all_managed_features if f in df.columns]
    if existing_features:
        df = df.drop(columns=existing_features)
        print(f"Removed {len(existing_features)} outdated features: {existing_features}")
    
    # === 3. 批量计算新特征 ===
    new_features = {}
    
    # 动量特征（使用清理后的数据计算）
   
    new_features['mom_21d'] = df['hist_returns'] .rolling(window=21, min_periods=1).sum()
    new_features['mom_42d'] = df['hist_returns'] .rolling(window=42, min_periods=1).sum()
    new_features['mom_63d'] = df['hist_returns'] .rolling(window=63, min_periods=1).sum()
    new_features['mom_126'] = df['hist_returns'] .rolling(window=126, min_periods=1).sum()
    # 情绪特征
    for col in groups.get('S', []):
        if col not in df.columns:
            continue
            
        shifted = df[col].shift(1)
        new_features[f"lag_{col}"] = (
            0.5 * shifted +
            0.3 * shifted.rolling(3, min_periods=1).mean() +
            0.2 * shifted.rolling(5, min_periods=1).mean()
        )
    
    # === 4. 一次性合并所有新特征 ===
    if new_features:
        features_df = pd.DataFrame(new_features, index=df.index)
        df = pd.concat([df, features_df], axis=1)
    
    return df

train=create_finance_features(train)
drop_cols = ['hist_returns','forward_returns', 'risk_free_rate']
drop_cols = [c for c in drop_cols if c in train.columns]


train.drop(columns=drop_cols, inplace=True, errors='ignore')
print(train.shape)

        


            

(9021, 104)


您提出了一个关键问题：多次运行特征工程函数确实会导致重复创建相同名称的特征列（如 lag_S1、lag_S2 等），这会造成：

列爆炸：每次运行新增一组特征列
内存浪费：存储重复数据
后续错误：模型训练时特征维度异常膨胀
根本原因分析
Pandas 在列名已存在时不会覆盖，而是：

第一次运行：创建 lag_S1
第二次运行：创建 lag_S1.1（自动添加后缀）
第三次运行：创建 lag_S1.2
→ 导致列名混乱和重复
 # 1. 确定需要创建的特征名（仅原始列对应的特征）
    target_features = [
        f"lag_{col}" 
        for col in groups.get('S', []) 
        if col in df.columns
    ]
    
    # 2. 删除已存在的旧特征（确保幂等性）
    existing_features = [f for f in target_features if f in df.columns]
    if existing_features:
        df = df.drop(columns=existing_features)
        print(f"Removed {len(existing_features)} outdated features: {existing_features}")
    

In [ ]:
#先复制一个，占点内存没什么

TARGET_COL = 'market_forward_excess_returns'
drop_cols = ['hist_returns','date_id', 'forward_returns', 'risk_free_rate', TARGET_COL]
drop_cols = [c for c in drop_cols if c in train.columns]

score_forward_returns=pd.Series(train.iloc[n_train:]['forward_returns'])
score_risk_free_rate=pd.Series(train.iloc[n_train:]['risk_free_rate'])


train.drop(columns=drop_cols, inplace=True, errors='ignore')

n_test = 300
n_train = len(train) - n_test

feature_cols = [col for col in train.columns if col not in drop_cols ]

x_train = train.iloc[:n_train, train.columns.isin(feature_cols)]
y_train = train.iloc[:n_train][TARGET_COL]
x_test = train.iloc[n_train:, train.columns.isin(feature_cols)]
y_test = train.iloc[n_train:][TARGET_COL]
train_score_forward_returns=pd.Series(train.iloc[:n_train]['forward_returns'])
train_score_risk_free_rate=pd.Series(train.iloc[:n_train]['risk_free_rate'])

print(f"nonsum:{x_test.notnull().sum()}")
print(f"nonsum:{x_train.notnull().sum()}")
print(y_train.head)

特征选择

In [ ]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
import xgboost as xgb

# ===== 1. 过滤式初筛 =====
var_selector = VarianceThreshold(threshold=0.01).fit(x_train)
x_train_var = pd.DataFrame(var_selector.transform(x_train), columns=x_train.columns[var_selector.get_support()])
x_test_var = pd.DataFrame(var_selector.transform(x_test), columns=x_test.columns[var_selector.get_support()])

# 互信息选择 (在原始尺度上计算!)
mi_scores = mutual_info_regression(x_train_var, y_train, random_state=42)
top_features = pd.Series(mi_scores, index=x_train_var.columns).sort_values(ascending=False).index[:50].tolist()

# ===== 2. 标准化选定特征 =====
scaler = StandardScaler().fit(x_train_var[top_features])  # 仅拟合选定的50个特征
x_train_scaled = pd.DataFrame(scaler.transform(x_train_var[top_features]), columns=top_features)
x_test_scaled = pd.DataFrame(scaler.transform(x_test_var[top_features]), columns=top_features)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()

# ===== 3. ElasticNet + XGBoost 融合 =====
# ElasticNet (抗共线性)
elastic = ElasticNetCV(l1_ratio=[0.1,0.3, 0.5,0.7, 0.9], cv=3, max_iter=10000,random_state=42).fit(x_train_scaled, y_train)
elastic_features = x_train_scaled.columns[np.abs(elastic.coef_) > 1e-5].tolist()

# XGBoost (非线性)
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42).fit(x_train_scaled, y_train)
xgb_features = pd.Series(xgb_model.feature_importances_, index=top_features).sort_values(ascending=False).index[:40].tolist()

# 融合选择Top30
candidate_features = list(set(elastic_features + xgb_features))
rank_df = pd.DataFrame(index=candidate_features)
rank_df['elastic_rank'] = pd.Series(elastic.coef_, index=top_features).reindex(candidate_features).abs().rank(ascending=False)
rank_df['xgb_rank'] = pd.Series(xgb_model.feature_importances_, index=top_features).reindex(candidate_features).rank(ascending=False)
rank_df['final_rank'] = 0.6 * rank_df['elastic_rank'] + 0.4 * rank_df['xgb_rank']
top30_features = rank_df.sort_values('final_rank').index[:30].tolist()

# ===== 4. 应用最终特征 =====
x_train_final = x_train_scaled[top30_features]
x_test_final = x_test_scaled[top30_features]

print(f"✅ 特征选择完成 | 最终特征: {len(top30_features)}")
print(f"Top5: {top30_features[:5]}")

# 保存筛选后数据集 (保持时间顺序!)
filtered_train = pd.concat([x_train_final, y_train], axis=1)
filtered_test = pd.concat([x_test_final, y_test], axis=1)
filtered_all = pd.concat([filtered_train, filtered_test])


前面特征工程已经分好训练集测试集，截取最后300行作为测试集，然后定义出来[x_train,y_train;x_test,y_test]
剩下的模型我们分为三步：
1.创建预测模型
==2.仓位大小映射函数！！！==
3.策略回测引擎
我们这里还是默认返回了一个处理后的df，命名依然为train

关于预测模型，
最终建模流程总结
输入：train 数据集包含：
feature_1 ... feature_30
y = market_forward_excess_returns（训练标签）
forward_returns（用于构造策略收益）
risk_free_rate（用于构造策略收益）
模型训练：
用 X → y 训练 LightGBM 回归器；
使用 TimeSeriesSplit 防止时序泄露。
预测后处理：
将 pred_y 通过 Sigmoid / 截断缩放 映射到 [0, 2] 得到 position。
验证与选模：
在验证集上用你提供的 score() 函数计算 adjusted Sharpe；
以此为指标选择模型和超参。
输出：
最终提交的 submission['prediction'] = position ∈ [0, 2]

In [ ]:

n= len(x_train)
split_idx = int(0.85 * n)  # 85% 训练，15% 验证（用于早停）
val_frac=int(0.15*n)

X_tr = x_train.iloc[:split_idx]
X_val = x_train.iloc[split_idx:]

y_tr = y_train.iloc[:split_idx]
y_val = y_train.iloc[split_idx:]
val_forward_returns=train_score_forward_returns[-val_frac:]
val_risk_free_rate=train_score_risk_free_rate[-val_frac:]

# ----------------------------
# 2. 定义 LightGBM 模型
# ----------------------------
params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 2000  # 将被早停截断
}

train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# ----------------------------
# 3. 训练模型（带早停）
# ----------------------------
model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(100)
    ]
)

# ----------------------------
# 4. 预测（raw output）
# ----------------------------
# 对训练集 & 测试集都预测（方便后续分析）
pred_train_raw = model.predict(x_train)   # 用于诊断
pred_test_raw = model.predict(x_test)     # 最终输出（待后处理）

# 转为 Series（保留索引）
pred_test_raw = pd.Series(pred_test_raw, index=x_test.index, name='pred_excess')


In [ ]:
# ----------------------------
# Step 0: 内联你提供的 score 函数（避免依赖）
# ----------------------------
MIN_INVESTMENT = 0.00
MAX_INVESTMENT = 2.00

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')
    
    # 使用 copy 避免修改原始数据
    sol = solution.copy()
    sol['position'] = submission['prediction']
    
    if sol['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {sol["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {sol["position"].min()} below minimum of {MIN_INVESTMENT}')
    
    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    
    # Strategy Sharpe
    strategy_excess_returns = sol['strategy_returns'] - sol['risk_free_rate']
    n = len(sol)
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = strategy_excess_cumulative ** (1 / n) - 1
    strategy_std = sol['strategy_returns'].std()
    
    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    
    # Market stats
    market_excess_returns = sol['forward_returns'] - sol['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = market_excess_cumulative ** (1 / n) - 1
    market_std = sol['forward_returns'].std()
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)
    
    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')
    
    # Volatility penalty
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2)
    vol_penalty = 1 + excess_vol
    
    # Return penalty
    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap ** 2) / 100
    
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

# ----------------------------
# Step 1: 仓位映射函数（可调）
# ----------------------------
def map_to_position(pred_excess: np.ndarray, method='sigmoid', k=10.0):
    """
    将预测的 excess return 映射到 [0, 2] 仓位。
    
    Parameters:
    - pred_excess: array of raw model predictions (y_hat)
    - method: 'sigmoid', 'clip_scale', or 'binary'
    - k: sigmoid 陡峭度（越大越接近阶跃）
    """
    if method == 'sigmoid':
        # Sigmoid 映射到 (0, 2)
        position = 2 / (1 + np.exp(-k * pred_excess))
    elif method == 'clip_scale':
        # 截断后线性缩放到 [0, 2]
        lower = np.percentile(pred_excess, 5)
        upper = np.percentile(pred_excess, 95)
        clipped = np.clip(pred_excess, lower, upper)
        position = 2 * (clipped - clipped.min()) / (clipped.max() - clipped.min() + 1e-8)
    elif method == 'binary':
        # 简单二值：正收益满仓，否则空仓
        position = np.where(pred_excess > 0, 2.0, 0.0)
    else:
        raise ValueError("method must be 'sigmoid', 'clip_scale', or 'binary'")
    
    # 确保严格在 [0, 2]
    position = np.clip(position, 0, 2)
    return position

# ----------------------------
# Step 2: 完整评估流程
# ----------------------------
def evaluate_model(
    pred_test_raw: pd.Series,
    forward_returns: pd.Series,
    risk_free_rate: pd.Series,
    mapping_method='sigmoid',
    k=10.0
) -> tuple[float, pd.Series]:
    """
    输入：
        pred_test_raw: 模型对 x_test 的原始预测 (y_hat)
        forward_returns: 对应的真实 forward_returns
        risk_free_rate: 对应的 risk_free_rate
    
    输出：
        adjusted_sharpe: 改进夏普比率
        position: 映射后的仓位（可用于分析）
    """
    # 确保索引对齐
    assert pred_test_raw.index.equals(forward_returns.index), "Index mismatch"
    assert pred_test_raw.index.equals(risk_free_rate.index), "Index mismatch"
    
    # Step A: 映射到仓位
    position = map_to_position(
        pred_test_raw.values,
        method=mapping_method,
        k=k
    )
    position = pd.Series(position, index=pred_test_raw.index, name='prediction')
    
    # Step B: 构造 solution 和 submission
    solution_df = pd.DataFrame({
        'forward_returns': forward_returns,
        'risk_free_rate': risk_free_rate
    })
    submission_df = pd.DataFrame({'prediction': position})
    
    # Step C: 调用 score
    try:
        adj_sharpe = score(solution_df, submission_df, row_id_column_name=None)
    except ParticipantVisibleError as e:
        print(f"Score error: {e}")
        adj_sharpe = -np.inf
    
    return adj_sharpe, position

# ----------------------------
# 示例用法（假设你已有以下变量）：
# - pred_test_raw: pd.Series
# - forward_returns_test: pd.Series (对应 x_test)
# - risk_free_rate_test: pd.Series (对应 x_test)
# ----------------------------
# adjusted_sharpe, position = evaluate_model(
#     pred_test_raw,
#     forward_returns_test,
#     risk_free_rate_test,
#     mapping_method='sigmoid',
#     k=15.0
# )
# print(f"Adjusted Sharpe: {adjusted_sharpe:.4f}")

消融实验开始调参

In [ ]:


# ------------------------------------------------------------------
# 3. 消融实验配置
# ------------------------------------------------------------------
# 超参网格（可按需扩展）
lgb_param_grid = [
    {'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8},
    {'learning_rate': 0.1,  'num_leaves': 31, 'feature_fraction': 0.8},
    {'learning_rate': 0.05, 'num_leaves': 63, 'feature_fraction': 0.7},
    {'learning_rate': 0.01, 'num_leaves': 15, 'feature_fraction': 0.9},
]

mapping_methods = ['sigmoid', 'clip_scale', 'binary']
k_values = [5, 10, 15, 20]  # 仅 sigmoid 使用

# 存储结果
results = []
best_score = -np.inf
best_config = None

# ------------------------------------------------------------------
# 4. 实验循环
# ------------------------------------------------------------------
for i, params in enumerate(lgb_param_grid):
    print(f"\n[Experiment {i+1}/{len(lgb_param_grid)}] Training with params: {params}")
    
    # 构建 LightGBM Dataset
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # 固定其他参数
    fixed_params = {
        'objective': 'regression',
        'metric': 'mse',
        'boosting_type': 'gbdt',
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'verbose': -1,
        'random_state': 42,
        'n_estimators': 2000
    }
    full_params = {**fixed_params, **params}
    
    # 训练模型（带早停）
    model = lgb.train(
        full_params,
        train_data,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(0)  # 静默
        ]
    )
    
    # 预测验证集 raw output
    pred_val_raw = model.predict(X_val)
    pred_val_raw = pd.Series(pred_val_raw, index=X_val.index)
    
    # 遍历映射策略
    for method in mapping_methods:
        if method == 'sigmoid':
            for k in k_values:
                position = map_to_position(pred_val_raw.values, method=method, k=k)
                submission_df = pd.DataFrame({'prediction': position}, index=X_val.index)
                solution_df = pd.DataFrame({
                    'forward_returns': val_forward_returns.values,
                    'risk_free_rate': val_risk_free_rate.values
                }, index=X_val.index)
                
                try:
                    val_score = score(solution_df, submission_df, row_id_column_name=None)
                except ParticipantVisibleError:
                    val_score = -np.inf
                
                # 记录
                config = {
                    'lgb_params': params,
                    'mapping_method': method,
                    'k': k,
                    'val_score': val_score
                }
                results.append(config)
                
                if val_score > best_score:
                    best_score = val_score
                    best_config = config.copy()
                    best_config['model'] = model  # 保留模型引用
                
                print(f"  → method={method}, k={k} → Adjusted Sharpe: {val_score:.4f}")
        else:
            position = map_to_position(pred_val_raw.values, method=method)
            submission_df = pd.DataFrame({'prediction': position}, index=X_val.index)
            solution_df = pd.DataFrame({
                'forward_returns': val_forward_returns.values,
                'risk_free_rate': val_risk_free_rate.values
            }, index=X_val.index)
            
            try:
                val_score = score(solution_df, submission_df, row_id_column_name=None)
            except ParticipantVisibleError:
                val_score = -np.inf
            
            config = {
                'lgb_params': params,
                'mapping_method': method,
                'k': None,
                'val_score': val_score
            }
            results.append(config)
            
            if val_score > best_score:
                best_score = val_score
                best_config = config.copy()
                best_config['model'] = model
            
            print(f"  → method={method} → Adjusted Sharpe: {val_score:.4f}")

# ------------------------------------------------------------------
# 5. 输出最佳结果
# ------------------------------------------------------------------
print("\n" + "="*60)
print("✅ Best Configuration on Validation Set:")
print(f"Adjusted Sharpe: {best_score:.4f}")
print(f"Model Params: {best_config['lgb_params']}")
print(f"Mapping Method: {best_config['mapping_method']}")
if best_config['mapping_method'] == 'sigmoid':
    print(f"Sigmoid k: {best_config['k']}")
print("="*60)

# 保存 results 到 DataFrame（可选）
results_df = pd.DataFrame(results)
results_df.to_csv('ablation_results.csv', index=False)
print("\nSaved all results to 'ablation_results.csv'")

最终测试结果

In [ ]:
from datetime import datetime
# ------------------------------------------------------------------
# 1. 确保 best_config 已从消融实验获取
# ------------------------------------------------------------------
# 假设 best_config 已存在，包含:
# - best_config['lgb_params']: 最佳超参
# - best_config['mapping_method']: 最佳映射方法
# - best_config['k']: 最佳k值（如果是sigmoid）
print(f"Best config from validation: {best_config}")

# ------------------------------------------------------------------
# 2. 用全训练集重新训练最终模型
# ------------------------------------------------------------------
print("\n🚀 Training final model on full training set...")

# 准备全训练集数据
train_data_full = lgb.Dataset(x_train, label=y_train)

# 合并固定参数 + 最佳参数
fixed_params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbose': -1,
    'random_state': 42,
    'n_estimators': 2000
}
final_params = {**fixed_params, **best_config['lgb_params']}

# 为早停准备一个小的内部验证集（取训练集最后10%）
n = len(x_train)
val_size = int(0.1 * n)
X_internal_val = x_train.iloc[-val_size:]
y_internal_val = y_train.iloc[-val_size:]
val_data_internal = lgb.Dataset(X_internal_val, label=y_internal_val, reference=train_data_full)

# 训练最终模型（带早停）
final_model = lgb.train(
    final_params,
    train_data_full,
    valid_sets=[val_data_internal],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(50)
    ]
)

# 保存模型（可选）
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f'final_model_{timestamp}.pkl'
joblib.dump(final_model, model_path)
print(f"💾 Model saved to {model_path}")

# ------------------------------------------------------------------
# 3. 测试集预测
# ------------------------------------------------------------------
print("\n🔍 Predicting on test set...")
pred_test_raw = final_model.predict(x_test)
pred_test_raw = pd.Series(pred_test_raw, index=x_test.index, name='pred_excess')

# ------------------------------------------------------------------
# 4. 仓位映射（使用最佳配置）
# ------------------------------------------------------------------
print("\n🔄 Mapping predictions to positions...")
if best_config['mapping_method'] == 'sigmoid':
    position_test = map_to_position(
        pred_test_raw.values,
        method='sigmoid',
        k=best_config['k']
    )
else:
    position_test = map_to_position(
        pred_test_raw.values,
        method=best_config['mapping_method']
    )

position_test = pd.Series(position_test, index=x_test.index, name='position')
print(f"Position stats: min={position_test.min():.4f}, max={position_test.max():.4f}, mean={position_test.mean():.4f}")

# ------------------------------------------------------------------
# 5. 构造 solution 和 submission 用于 score
# ------------------------------------------------------------------
# 确保你有测试集对应的 forward_returns_test 和 risk_free_rate_test
# 假设它们已经存在且与 x_test 索引对齐

solution_test = pd.DataFrame({
    'forward_returns': score_forward_returns,
    'risk_free_rate': score_risk_free_rate
}, index=x_test.index)

submission_test = pd.DataFrame({
    'prediction': position_test
}, index=x_test.index)

# ------------------------------------------------------------------
# 6. 调用 score 函数计算最终分数
# ------------------------------------------------------------------
print("\n📊 Calculating final Adjusted Sharpe Ratio...")
try:
    final_score = score(solution_test, submission_test, row_id_column_name=None)
    print(f"\n🎉 FINAL TEST ADJUSTED SHARPE RATIO: {final_score:.4f}")
except ParticipantVisibleError as e:
    print(f"❌ Score calculation failed: {e}")
    final_score = -np.inf

# ------------------------------------------------------------------
# 7. 保存结果与分析
# ------------------------------------------------------------------
# 7.1 保存仓位
positions_df = pd.DataFrame({
    'date': x_test.index,  # 假设索引是日期
    'position': position_test,
    'pred_excess': pred_test_raw,
    'forward_returns': forward_returns_test,
    'risk_free_rate': risk_free_rate_test
})
positions_df.to_csv(f'test_positions_{timestamp}.csv', index=False)
print(f"💾 Positions saved to test_positions_{timestamp}.csv")

# 7.2 保存最终分数
with open(f'final_score_{timestamp}.txt', 'w') as f:
    f.write(f"Final Adjusted Sharpe Ratio: {final_score:.4f}\n")
    f.write(f"Model params: {final_params}\n")
    f.write(f"Mapping method: {best_config['mapping_method']}\n")
    if best_config['mapping_method'] == 'sigmoid':
        f.write(f"Sigmoid k: {best_config['k']}\n")
print(f"💾 Score report saved to final_score_{timestamp}.txt")


def plot_strategy_analysis(positions_df, final_score):
    """生成策略分析图表"""
    plt.figure(figsize=(15, 10))
    
    # 2. 策略收益 vs 市场收益
    positions_df['strategy_returns'] = (
        positions_df['risk_free_rate'] * (1 - positions_df['position']) + 
        positions_df['position'] * positions_df['forward_returns']
    )
    positions_df['market_returns'] = positions_df['forward_returns']
    
    # 计算累计收益
    positions_df['cum_strategy'] = (1 + positions_df['strategy_returns']).cumprod()
    positions_df['cum_market'] = (1 + positions_df['market_returns']).cumprod()
    
    plt.subplot(3, 1, 2)
    plt.plot(positions_df['date'], positions_df['cum_strategy'], label='Strategy', linewidth=2)
    plt.plot(positions_df['date'], positions_df['cum_market'], label='Market (S&P 500)', linestyle='--')
    plt.title('Cumulative Returns: Strategy vs Market', fontsize=14)
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # 3. 仓位 vs 预测信号
    plt.subplot(3, 1, 3)
    plt.scatter(positions_df['pred_excess'], positions_df['position'], 
                alpha=0.6, s=10, color='purple')
    plt.title('Position vs Predicted Excess Return', fontsize=14)
    plt.xlabel('Predicted Excess Return')
    plt.ylabel('Position')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plot_path = f'strategy_analysis_{timestamp}.png'
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    print(f"📈 Strategy analysis plot saved to {plot_path}")
    plt.close()

# 生成分析图表
try:
    plot_strategy_analysis(positions_df, final_score)
except Exception as e:
    print(f"⚠️ Failed to generate plots: {e}")

# ------------------------------------------------------------------
# 9. 特征重要性分析（可选）
# ------------------------------------------------------------------
if hasattr(final_model, 'feature_importance'):
    # 获取特征重要性
    feature_importance = pd.DataFrame({
        'feature': x_train.columns,
        'importance': final_model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    
    # 保存
    feature_importance.to_csv(f'feature_importance_{timestamp}.csv', index=False)
    print(f"📊 Feature importance saved to feature_importance_{timestamp}.csv")
    
    # 绘制Top 15
    plt.figure(figsize=(12, 8))
    sns.barplot(
        x='importance', 
        y='feature', 
        data=feature_importance.head(15),
        palette='viridis'
    )
    plt.title('Top 15 Features by Importance (Gain)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'feature_importance_plot_{timestamp}.png', dpi=120)
    plt.close()
    print(f"📈 Feature importance plot saved")

# ------------------------------------------------------------------
# 10. 完成提示
# ------------------------------------------------------------------
print("\n" + "="*60)
print("✅ FINAL EVALUATION COMPLETE!")
print(f"🎯 Final Test Adjusted Sharpe Ratio: {final_score:.4f}")
print(f"💾 All results saved with timestamp: {timestamp}")
print("="*60)